# Neural Network Forecasting with Load Factor Feature

Notebook 15b tested whether `load_factor_lag1` improves the forecasting model using traditional models (Ridge, RF, XGBoost). The results were mixed: XGBoost classification benefited moderately while Ridge and RF regression showed minimal change.

The hypothesis here is that the neural network's nonlinear modelling capacity may extract signal from load factor in the forecasting context that traditional models could not. Since forecasting lacks same-month weather features, load factor may play a relatively larger role by providing demand-side information absent from the lagged delay features.

## Approach

Three load factor transforms are tested (lag1 only, since same-month load factor is unavailable at forecast time):
- **Raw:** `load_factor_lag1` (linear)
- **Exp:** `load_factor_lag1_exp` = exp(load_factor_lag1)
- **-Log(1-x):** `load_factor_lag1_log_inv` = -log(1 - load_factor_lag1)

No interaction terms with weather are included, as same-month weather features are unavailable for forecasting.

## Reference (notebook 15b, forecasting with lag12)

| Model | R² | F1 |
|-------|----|----|   
| Ridge | 0.492 | - |
| Random Forest | 0.506 | - |
| XGBoost | - | 0.737 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                             f1_score, roc_auc_score, precision_score, recall_score)

np.random.seed(42)

try:
    import xgboost as xgb
    HAS_XGB = True
    print('XGBoost available')
except ImportError:
    HAS_XGB = False
    print('XGBoost not installed')

try:
    import tensorflow as tf
    tf.random.set_seed(42)
    from tensorflow import keras
    HAS_TF = True
    print('TensorFlow {} available'.format(tf.__version__))
except ImportError:
    HAS_TF = False
    print('TensorFlow not installed')

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.linewidth'] = 1.5

%matplotlib inline

## 1. Data Preparation

The training data and load factor data are loaded following notebooks 15b and 12, respectively. The load factor is sourced from the BITRE Monthly Airline Performance dataset, which provides a single industry-aggregate value per month. All airline-route combinations in a given month therefore receive the same load factor value.

In [ ]:
df = pd.read_csv('../data/processed/ml_training_data_multiroute_hols.csv')

df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month_num'] = df['year_month_dt'].dt.month
df['year'] = df['year'].astype(int)
df['airline_route'] = df['airline'] + '_' + df['departing_port'] + '_' + df['arriving_port']
df['route'] = df['departing_port'] + '_' + df['arriving_port']
df = df.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)

print('Shape: {}'.format(df.shape))
print('Date range: {} to {}'.format(df['year_month'].min(), df['year_month'].max()))

In [ ]:
# Load BITRE Monthly Airline Performance and extract load factor
activity_candidates = [
    '../data/raw/monthly-airline-performance-november-2025.xlsx',
    '../data/raw/monthly-airline-performance-october-2025.xlsx',
]
ACTIVITY_FILE = next((f for f in activity_candidates if os.path.exists(f)), None)
if ACTIVITY_FILE is None:
    raise FileNotFoundError('Monthly airline performance file not found.')
SHEET_NAME = 'Domestic airlines'

df_activity = pd.read_excel(ACTIVITY_FILE, sheet_name=SHEET_NAME, header=None, skiprows=8)
df_activity.columns = [
    'year', 'month_name', 'hours_flown', 'aircraft_km_flown_000', 'aircraft_departures',
    'total_rev_pax_ud', 'freight_tonnes_ud', 'mail_tonnes_ud',
    'total_rev_pax_tob', 'total_rev_pax_tob_inc_intl',
    'freight_tonnes_tob', 'mail_tonnes_tob',
    'total_rpk_000', 'pax_tonne_km_000', 'freight_tonne_km_000',
    'mail_tonne_km_000', 'total_tonne_km_000',
    'available_seat_km_000', 'available_tonne_km_000', 'available_seats_000',
    'pax_load_factor_pct', 'weight_load_factor_pct',
    'total_charter_pax_tob', 'charter_aircraft_departures'
]

df_activity['year_numeric'] = pd.to_numeric(df_activity['year'], errors='coerce')
valid_months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'June',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

df_act = df_activity[
    df_activity['year_numeric'].notna() &
    df_activity['month_name'].isin(valid_months)
].copy()

df_act['year'] = df_act['year_numeric'].astype(int)
df_act = df_act.drop(columns=['year_numeric'])
df_act['month_name'] = df_act['month_name'].replace({'June': 'Jun'})

month_map = {'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
             'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12}
df_act['month_num'] = df_act['month_name'].map(month_map)
df_act['year_month'] = df_act['year'].astype(str) + '-' + df_act['month_num'].astype(str).str.zfill(2)
df_act['load_factor'] = pd.to_numeric(df_act['pax_load_factor_pct'], errors='coerce') / 100.0

# Exclude COVID period
covid_mask = (df_act['year'] == 2020) & (df_act['month_num'] >= 4)
df_act = df_act[~covid_mask].copy()
df_act = df_act[df_act['year'] >= 2009].copy()

df_lf = df_act[['year_month', 'load_factor']].groupby('year_month', as_index=False).mean()
df = df.merge(df_lf, on='year_month', how='left')

matched = df['load_factor'].notna().sum()
match_rate = matched / len(df) * 100

print('Load factor source: {}'.format(ACTIVITY_FILE))
print('Load factor records: {}'.format(len(df_lf)))
print('Match rate: {:.1f}% ({}/{})'.format(match_rate, matched, len(df)))

In [ ]:
# Filter low-volume airline-routes
volume_threshold = 40
airline_route_volume = df.groupby('airline_route')['sectors_scheduled'].mean()
low_volume = airline_route_volume[airline_route_volume < volume_threshold]
high_volume_ar = airline_route_volume[airline_route_volume >= volume_threshold].index.tolist()
df_filtered = df[df['airline_route'].isin(high_volume_ar)].copy()

print('Volume threshold: {} flights/month'.format(volume_threshold))
print('Excluded airline-routes: {}'.format(len(low_volume)))
print('Records after filtering: {:,} (from {:,})'.format(len(df_filtered), len(df)))

In [ ]:
# Exclude anomalous routes
anomalous_routes = ['Melbourne_Hobart', 'Adelaide_Sydney', 'Perth_Brisbane']
before = len(df_filtered)
df_filtered = df_filtered[~df_filtered['route'].isin(anomalous_routes)].copy()
print('Excluded anomalous routes: {}'.format(anomalous_routes))
print('Records: {:,} -> {:,}'.format(before, len(df_filtered)))

## 2. Feature Engineering

Standard lag features and cyclical month encoding are created following the established pipeline (notebook 15a). No same-month weather features are computed, as they are unavailable at forecast time. Load factor transforms are then computed following the methodology of notebook 12 (lag1 only).

In [ ]:
# Standard lag features (no same-month weather for forecasting)
df_filtered['delay_rate_lag1'] = df_filtered.groupby('airline_route')['delay_rate'].shift(1)
df_filtered['delay_rate_lag2'] = df_filtered.groupby('airline_route')['delay_rate'].shift(2)
df_filtered['delay_rate_gradient'] = df_filtered['delay_rate_lag1'] - df_filtered['delay_rate_lag2']
df_filtered['delay_rate_lag12'] = df_filtered.groupby('airline_route')['delay_rate'].shift(12)

# Cyclical month encoding
df_filtered['month_sin'] = np.sin(2 * np.pi * df_filtered['month_num'] / 12)
df_filtered['month_cos'] = np.cos(2 * np.pi * df_filtered['month_num'] / 12)

# One-hot encoding
airline_dummies = pd.get_dummies(df_filtered['airline'], prefix='airline')
df_filtered = pd.concat([df_filtered, airline_dummies], axis=1)
airline_cols = sorted(airline_dummies.columns.tolist())

route_dummies = pd.get_dummies(df_filtered['route'], prefix='route')
df_filtered = pd.concat([df_filtered, route_dummies], axis=1)
route_cols = sorted(route_dummies.columns.tolist())

print('Airlines: {}'.format(len(airline_cols)))
print('Routes:   {}'.format(len(route_cols)))

In [ ]:
# Load factor features: lag1 transforms only (no same-month, no weather interactions)

# Lag1 of load factor (available at forecast time)
df_filtered['load_factor_lag1'] = df_filtered.groupby('airline_route')['load_factor'].shift(1)

# Exponential transform of lag1
df_filtered['load_factor_lag1_exp'] = np.exp(df_filtered['load_factor_lag1'])

# -Log(1-x) transform of lag1 (clip to avoid log(0))
df_filtered['load_factor_lag1_log_inv'] = -np.log(1 - df_filtered['load_factor_lag1'].clip(upper=0.99))

print('Load factor features created:')
lf_features = ['load_factor_lag1', 'load_factor_lag1_exp', 'load_factor_lag1_log_inv']
for feat in lf_features:
    valid = df_filtered[feat].notna().sum()
    print('  {:<35} valid: {:,}'.format(feat, valid))

## 3. Exploratory Analysis of Load Factor

A brief examination of load factor lag1 characteristics is provided to contextualise the feature before model evaluation. Since this is industry-aggregate data lagged by one month, the correlation with delay rate reflects shared seasonal patterns with a one-month offset.

In [ ]:
# Distribution and time series
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
df_filtered['load_factor_lag1'].dropna().hist(bins=30, ax=ax, color='steelblue', alpha=0.8, edgecolor='white')
ax.set_xlabel('Load Factor (Lag1)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Load Factor Lag1')
ax.axvline(df_filtered['load_factor_lag1'].dropna().mean(), color='red', linestyle='--',
           label='Mean: {:.3f}'.format(df_filtered['load_factor_lag1'].dropna().mean()))
ax.legend()

ax = axes[1]
lf_ts = df_filtered.groupby('year_month_dt')['load_factor_lag1'].mean()
ax.plot(lf_ts.index, lf_ts.values, color='steelblue', linewidth=1.5)
ax.set_xlabel('Date')
ax.set_ylabel('Load Factor (Lag1)')
ax.set_title('Load Factor Lag1 Over Time (Monthly)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation analysis
corr_features = ['delay_rate', 'delay_rate_lag1', 'delay_rate_lag12', 'sectors_scheduled',
                 'delay_rate_gradient',
                 'load_factor_lag1', 'load_factor_lag1_exp', 'load_factor_lag1_log_inv']

valid_data = df_filtered[corr_features].dropna()
corrs = valid_data.corr()['delay_rate'].drop('delay_rate').sort_values(ascending=False)

print('Correlation with delay_rate:')
print('-' * 55)
for feat, val in corrs.items():
    marker = ' <-- LF' if 'load_factor' in feat else ''
    print('  {:<40} {:+.4f}{}'.format(feat, val, marker))

# Check multicollinearity between LF variants and existing features
print()
print('Multicollinearity check (|r| > 0.7):')
lf_cols = [c for c in corr_features if 'load_factor' in c]
existing_cols = ['delay_rate_lag1', 'delay_rate_lag12', 'sectors_scheduled']
found_high = False
for lf in lf_cols:
    for ex in existing_cols:
        r = valid_data[lf].corr(valid_data[ex])
        if abs(r) > 0.7:
            print('  {} x {}: r = {:.4f}'.format(lf, ex, r))
            found_high = True
if not found_high:
    print('  No high multicollinearity detected.')

## 4. Data Preparation for Modelling

All models are trained on the subset of rows where lag1, lag2, gradient, lag12, and load_factor_lag1 are all available. This ensures fair comparison across all variants.

In [ ]:
# Drop rows with missing values in required columns
required_cols = ['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient',
                 'delay_rate_lag12', 'load_factor_lag1']

n_before = len(df_filtered)
n_after_lags = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient']).shape[0]
n_after_lag12 = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient', 'delay_rate_lag12']).shape[0]
df_clean = df_filtered.dropna(subset=required_cols).copy()
n_final = len(df_clean)

print('Data availability:')
print('  After filtering:          {:,}'.format(n_before))
print('  After lag1/lag2/gradient:  {:,} (-{:,})'.format(n_after_lags, n_before - n_after_lags))
print('  After lag12:              {:,} (-{:,}, {:.1f}%)'.format(
    n_after_lag12, n_after_lags - n_after_lag12,
    (n_after_lags - n_after_lag12) / n_after_lags * 100))
print('  After load_factor_lag1:   {:,} (-{:,}, {:.1f}%)'.format(
    n_final, n_after_lag12 - n_final,
    (n_after_lag12 - n_final) / n_after_lag12 * 100 if n_after_lag12 > 0 else 0))

In [ ]:
# Train/validation/test split
train_mask = ((df_clean['year'] <= 2017) | (df_clean['year'] == 2023))
val_mask = ((df_clean['year'] == 2018) | (df_clean['year'] == 2024))
test_mask = ((df_clean['year'] == 2019) | (df_clean['year'] >= 2025))

print('Train: {:,}  (years 2010-2017, 2023)'.format(train_mask.sum()))
print('Val:   {:,}  (years 2018, 2024)'.format(val_mask.sum()))
print('Test:  {:,}  (years 2019, 2025+)'.format(test_mask.sum()))

In [ ]:
# Define feature sets (no weather features for forecasting)
base_features = (airline_cols + route_cols
                 + ['month_sin', 'month_cos', 'delay_rate_lag1', 'sectors_scheduled'])
non_weather_features = ['delay_rate_gradient', 'n_public_holidays_total', 'pct_school_holiday']

FEATURES_BASELINE = base_features + ['delay_rate_lag12'] + non_weather_features

# Each LF variant adds exactly 1 feature (lag1 only, no same-month or interaction terms)
FEATURES_LF_RAW = FEATURES_BASELINE + ['load_factor_lag1']
FEATURES_LF_EXP = FEATURES_BASELINE + ['load_factor_lag1_exp']
FEATURES_LF_LOG = FEATURES_BASELINE + ['load_factor_lag1_log_inv']

FEATURE_SETS = {
    'Baseline (Lag12)': FEATURES_BASELINE,
    'LF Raw': FEATURES_LF_RAW,
    'LF Exp': FEATURES_LF_EXP,
    'LF -Log(1-x)': FEATURES_LF_LOG,
}

for name, feats in FEATURE_SETS.items():
    print('{:<20} {} features'.format(name, len(feats)))

## 5. Helper Functions

Model construction and evaluation functions are defined following the same architecture and hyperparameters as notebook 17b. The neural network uses a 32 → 16 → 1 feedforward architecture with ReLU activations, 30% dropout, Adam optimiser (learning rate 0.001), and early stopping with patience 20. Traditional model hyperparameters follow the forecasting convention from notebook 15b (Ridge alpha=100).

In [ ]:
def build_nn(input_dim, output_activation='linear'):
    """Build feedforward NN: 32 -> 16 -> 1."""
    model = keras.Sequential([
        keras.layers.Dense(32, activation='relu', input_shape=(input_dim,)),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(16, activation='relu'),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(1, activation=output_activation),
    ])
    return model


def train_traditional_models(df, features, train_mask, val_mask, test_mask):
    """Train Ridge, RF, and XGBoost. Return metrics."""
    X_train = df.loc[train_mask, features].values
    X_val = df.loc[val_mask, features].values
    X_test = df.loc[test_mask, features].values

    y_train = df.loc[train_mask, 'delay_rate'].values
    y_test = df.loc[test_mask, 'delay_rate'].values

    y_train_clf = df.loc[train_mask, 'is_high_delay'].values
    y_val_clf = df.loc[val_mask, 'is_high_delay'].values
    y_test_clf = df.loc[test_mask, 'is_high_delay'].values

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    results = {}

    # Ridge (alpha=100 for forecasting convention)
    ridge = Ridge(alpha=100)
    ridge.fit(X_train_scaled, y_train)
    ridge_pred = ridge.predict(X_test_scaled)
    results['ridge_r2'] = r2_score(y_test, ridge_pred)
    results['ridge_rmse'] = np.sqrt(mean_squared_error(y_test, ridge_pred))

    # Random Forest Regression
    rf = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5,
                               random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    results['rf_r2'] = r2_score(y_test, rf_pred)
    results['rf_rmse'] = np.sqrt(mean_squared_error(y_test, rf_pred))
    results['rf_model'] = rf

    # XGBoost Classification
    if HAS_XGB:
        xgb_clf = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                     min_child_weight=5, random_state=42, n_jobs=-1)
        xgb_clf.fit(X_train, y_train_clf, eval_set=[(X_val, y_val_clf)], verbose=False)
        xgb_pred = xgb_clf.predict(X_test)
        results['xgb_f1'] = f1_score(y_test_clf, xgb_pred)
    else:
        results['xgb_f1'] = np.nan

    return results


def train_nn_regression(df, features, train_mask, val_mask, test_mask, verbose=0):
    """Train NN for regression. Return model, history, metrics."""
    X_train = df.loc[train_mask, features].values
    X_val = df.loc[val_mask, features].values
    X_test = df.loc[test_mask, features].values

    y_train = df.loc[train_mask, 'delay_rate'].values
    y_val = df.loc[val_mask, 'delay_rate'].values
    y_test = df.loc[test_mask, 'delay_rate'].values

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    model = build_nn(len(features), output_activation='linear')
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss='mse', metrics=['mae'])

    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=20, restore_best_weights=True)

    history = model.fit(X_train_scaled, y_train,
                        validation_data=(X_val_scaled, y_val),
                        epochs=500, batch_size=64, callbacks=[early_stop],
                        verbose=verbose)

    best_epoch = early_stop.best_epoch if hasattr(early_stop, 'best_epoch') else len(history.history['loss']) - 20
    preds = model.predict(X_test_scaled, verbose=0).flatten()

    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)

    return {
        'model': model, 'history': history, 'best_epoch': best_epoch,
        'preds': preds, 'y_test': y_test, 'r2': r2, 'rmse': rmse, 'mae': mae,
        'scaler': scaler,
    }


def train_nn_classification(df, features, train_mask, val_mask, test_mask, verbose=0):
    """Train NN for classification. Return model, history, metrics."""
    X_train = df.loc[train_mask, features].values
    X_val = df.loc[val_mask, features].values
    X_test = df.loc[test_mask, features].values

    y_train = df.loc[train_mask, 'is_high_delay'].values
    y_val = df.loc[val_mask, 'is_high_delay'].values
    y_test = df.loc[test_mask, 'is_high_delay'].values

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    model = build_nn(len(features), output_activation='sigmoid')
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss='binary_crossentropy', metrics=['accuracy'])

    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=20, restore_best_weights=True)

    history = model.fit(X_train_scaled, y_train,
                        validation_data=(X_val_scaled, y_val),
                        epochs=500, batch_size=64, callbacks=[early_stop],
                        verbose=verbose)

    best_epoch = early_stop.best_epoch if hasattr(early_stop, 'best_epoch') else len(history.history['loss']) - 20
    proba = model.predict(X_test_scaled, verbose=0).flatten()
    preds = (proba >= 0.5).astype(int)

    f1 = f1_score(y_test, preds)
    auc = roc_auc_score(y_test, proba)
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)

    return {
        'model': model, 'history': history, 'best_epoch': best_epoch,
        'preds': preds, 'proba': proba, 'y_test': y_test,
        'f1': f1, 'auc': auc, 'precision': precision, 'recall': recall,
    }


print('Helper functions defined.')

## 6. Model Training

All four feature set variants are evaluated using Ridge, Random Forest, XGBoost, and the neural network. Each model is trained on the identical data subset for fair comparison. The baseline (36-feature lag12 forecasting set from notebook 15a) serves as the reference against which load factor variants are measured.

In [ ]:
# Train traditional models for all variants
trad_results = {}
for name, feats in FEATURE_SETS.items():
    print('Training traditional models: {}...'.format(name))
    trad_results[name] = train_traditional_models(df_clean, feats, train_mask, val_mask, test_mask)

print()
print('Traditional Model Results:')
print('{:<20} {:>10} {:>10} {:>10}'.format('Variant', 'Ridge R\u00b2', 'RF R\u00b2', 'XGB F1'))
print('-' * 52)
baseline_trad = trad_results['Baseline (Lag12)']
for name, r in trad_results.items():
    xgb_str = '{:.4f}'.format(r['xgb_f1']) if not np.isnan(r.get('xgb_f1', np.nan)) else 'N/A'
    print('{:<20} {:>10.4f} {:>10.4f} {:>10}'.format(name, r['ridge_r2'], r['rf_r2'], xgb_str))

print()
print('Deltas vs Baseline:')
for name, r in trad_results.items():
    if name == 'Baseline (Lag12)':
        continue
    d_ridge = r['ridge_r2'] - baseline_trad['ridge_r2']
    d_rf = r['rf_r2'] - baseline_trad['rf_r2']
    d_xgb = r['xgb_f1'] - baseline_trad['xgb_f1'] if not np.isnan(r.get('xgb_f1', np.nan)) else 0
    print('  {:<18} Ridge {:+.4f}, RF {:+.4f}, XGB F1 {:+.4f}'.format(name, d_ridge, d_rf, d_xgb))

In [ ]:
# Train NN regression for all variants
nn_reg_results = {}
for name, feats in FEATURE_SETS.items():
    print('Training NN regression: {}...'.format(name))
    nn_reg_results[name] = train_nn_regression(df_clean, feats, train_mask, val_mask, test_mask)
    r = nn_reg_results[name]
    print('  Best epoch: {}, R\u00b2: {:.4f}, RMSE: {:.4f}'.format(r['best_epoch'], r['r2'], r['rmse']))
    print()

print('NN Regression Summary:')
print('{:<20} {:>10} {:>10} {:>10}'.format('Variant', 'R\u00b2', 'RMSE', 'MAE'))
print('-' * 52)
for name, r in nn_reg_results.items():
    print('{:<20} {:>10.4f} {:>10.4f} {:>10.4f}'.format(name, r['r2'], r['rmse'], r['mae']))

In [ ]:
# Train NN classification for all variants
nn_clf_results = {}
for name, feats in FEATURE_SETS.items():
    print('Training NN classification: {}...'.format(name))
    nn_clf_results[name] = train_nn_classification(df_clean, feats, train_mask, val_mask, test_mask)
    r = nn_clf_results[name]
    print('  Best epoch: {}, F1: {:.4f}, AUC: {:.4f}'.format(r['best_epoch'], r['f1'], r['auc']))
    print()

print('NN Classification Summary:')
print('{:<20} {:>10} {:>10} {:>10} {:>10}'.format('Variant', 'F1', 'AUC', 'Precision', 'Recall'))
print('-' * 62)
for name, r in nn_clf_results.items():
    print('{:<20} {:>10.4f} {:>10.4f} {:>10.4f} {:>10.4f}'.format(
        name, r['f1'], r['auc'], r['precision'], r['recall']))

## 7. Results Comparison

The complete results are presented across all models and feature set variants. The key question is whether any load factor transform provides a meaningful improvement over the lag12 forecasting baseline, and specifically whether the neural network benefits more than traditional models.

In [ ]:
# Comprehensive comparison table
print('=' * 90)
print('COMPREHENSIVE RESULTS COMPARISON')
print('=' * 90)
print()

header = '{:<20} {:>10} {:>10} {:>10} {:>10} {:>10} {:>10}'.format(
    'Variant', 'Ridge R\u00b2', 'RF R\u00b2', 'NN R\u00b2', 'XGB F1', 'NN F1', 'NN AUC')
print(header)
print('-' * 90)

for name in FEATURE_SETS:
    tr = trad_results[name]
    nr = nn_reg_results[name]
    nc = nn_clf_results[name]
    xgb_str = '{:.4f}'.format(tr['xgb_f1']) if not np.isnan(tr.get('xgb_f1', np.nan)) else 'N/A'
    print('{:<20} {:>10.4f} {:>10.4f} {:>10.4f} {:>10} {:>10.4f} {:>10.4f}'.format(
        name, tr['ridge_r2'], tr['rf_r2'], nr['r2'], xgb_str, nc['f1'], nc['auc']))

print()
print('DELTAS vs BASELINE:')
print('-' * 90)
bl_tr = trad_results['Baseline (Lag12)']
bl_nr = nn_reg_results['Baseline (Lag12)']
bl_nc = nn_clf_results['Baseline (Lag12)']

for name in list(FEATURE_SETS.keys())[1:]:
    tr = trad_results[name]
    nr = nn_reg_results[name]
    nc = nn_clf_results[name]
    d_xgb = tr['xgb_f1'] - bl_tr['xgb_f1'] if not np.isnan(tr.get('xgb_f1', np.nan)) else 0
    print('{:<20} {:>+10.4f} {:>+10.4f} {:>+10.4f} {:>+10.4f} {:>+10.4f} {:>+10.4f}'.format(
        name,
        tr['ridge_r2'] - bl_tr['ridge_r2'],
        tr['rf_r2'] - bl_tr['rf_r2'],
        nr['r2'] - bl_nr['r2'],
        d_xgb,
        nc['f1'] - bl_nc['f1'],
        nc['auc'] - bl_nc['auc']))

In [ ]:
# Bar chart: regression R\u00b2 and classification F1
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

variant_names = list(FEATURE_SETS.keys())
x = np.arange(len(variant_names))
width = 0.22

# Regression R\u00b2
ax = axes[0]
ridge_vals = [trad_results[n]['ridge_r2'] for n in variant_names]
rf_vals = [trad_results[n]['rf_r2'] for n in variant_names]
nn_vals = [nn_reg_results[n]['r2'] for n in variant_names]

bars1 = ax.bar(x - width, ridge_vals, width, label='Ridge', color='#3498db', alpha=0.85)
bars2 = ax.bar(x, rf_vals, width, label='Random Forest', color='#27ae60', alpha=0.85)
bars3 = ax.bar(x + width, nn_vals, width, label='Neural Network', color='#9b59b6', alpha=0.85)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                '{:.3f}'.format(bar.get_height()), ha='center', va='bottom', fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(variant_names, rotation=15, ha='right')
ax.set_ylabel('R\u00b2')
ax.set_title('Regression Performance')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, max(max(ridge_vals), max(rf_vals), max(nn_vals)) * 1.15)

# Classification F1
ax = axes[1]
xgb_vals = [trad_results[n]['xgb_f1'] for n in variant_names]
nn_f1_vals = [nn_clf_results[n]['f1'] for n in variant_names]

bars1 = ax.bar(x - width / 2, xgb_vals, width, label='XGBoost', color='#e74c3c', alpha=0.85)
bars2 = ax.bar(x + width / 2, nn_f1_vals, width, label='Neural Network', color='#9b59b6', alpha=0.85)

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                '{:.3f}'.format(bar.get_height()), ha='center', va='bottom', fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(variant_names, rotation=15, ha='right')
ax.set_ylabel('F1 Score')
ax.set_title('Classification Performance')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, max(max(xgb_vals), max(nn_f1_vals)) * 1.15)

plt.tight_layout()
plt.show()

In [ ]:
# NN-specific comparison
print('=' * 70)
print('NEURAL NETWORK: LOAD FACTOR IMPACT')
print('=' * 70)
print()

print('NN REGRESSION:')
bl = nn_reg_results['Baseline (Lag12)']
for name in list(FEATURE_SETS.keys()):
    r = nn_reg_results[name]
    if name == 'Baseline (Lag12)':
        print('  {:<18} R\u00b2 = {:.4f}, RMSE = {:.4f}'.format(name, r['r2'], r['rmse']))
    else:
        print('  {:<18} R\u00b2 = {:.4f} ({:+.4f}), RMSE = {:.4f} ({:+.4f})'.format(
            name, r['r2'], r['r2'] - bl['r2'], r['rmse'], r['rmse'] - bl['rmse']))

print()
print('NN CLASSIFICATION:')
bl_c = nn_clf_results['Baseline (Lag12)']
for name in list(FEATURE_SETS.keys()):
    r = nn_clf_results[name]
    if name == 'Baseline (Lag12)':
        print('  {:<18} F1 = {:.4f}, AUC = {:.4f}'.format(name, r['f1'], r['auc']))
    else:
        print('  {:<18} F1 = {:.4f} ({:+.4f}), AUC = {:.4f} ({:+.4f})'.format(
            name, r['f1'], r['f1'] - bl_c['f1'], r['auc'], r['auc'] - bl_c['auc']))

# Identify best LF variant for NN
lf_names = [n for n in FEATURE_SETS if n != 'Baseline (Lag12)']
best_lf_reg = max(lf_names, key=lambda n: nn_reg_results[n]['r2'])
best_lf_clf = max(lf_names, key=lambda n: nn_clf_results[n]['f1'])
print()
print('Best LF variant (NN regression): {}'.format(best_lf_reg))
print('Best LF variant (NN classification): {}'.format(best_lf_clf))

## 8. Training Curves

The training and validation loss curves for the neural network regression model are compared across feature set variants. Convergence behaviour and optimal epoch selection provide insight into whether load factor features add useful signal or introduce noise.

In [ ]:
# Training curves for NN regression (2x2 grid)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for idx, (name, r) in enumerate(nn_reg_results.items()):
    ax = axes[idx // 2][idx % 2]
    h = r['history'].history
    epochs = range(1, len(h['loss']) + 1)

    ax.plot(epochs, h['loss'], label='Train Loss', color='steelblue', linewidth=1.5)
    ax.plot(epochs, h['val_loss'], label='Val Loss', color='#e74c3c', linewidth=1.5)
    ax.axvline(r['best_epoch'], color='green', linestyle='--', alpha=0.7,
               label='Best Epoch: {}'.format(r['best_epoch']))

    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title('{} (R\u00b2={:.4f})'.format(name, r['r2']))
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('NN Regression Training Curves (Forecasting)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 9. Feature Importance

Feature importance is assessed through Random Forest impurity-based importance and neural network first-layer weight magnitudes. The ranking of load factor lag1 features relative to established predictors (lag1, lag12) indicates whether they provide meaningful signal in the forecasting context.

In [ ]:
# RF feature importance for each LF variant
lf_feature_names = {
    'LF Raw': {'load_factor_lag1'},
    'LF Exp': {'load_factor_lag1_exp'},
    'LF -Log(1-x)': {'load_factor_lag1_log_inv'},
}

for name in lf_names:
    rf_model = trad_results[name]['rf_model']
    feats = FEATURE_SETS[name]
    imp_df = pd.DataFrame({
        'Feature': feats,
        'Importance': rf_model.feature_importances_
    }).sort_values('Importance', ascending=False)

    lf_set = lf_feature_names[name]
    print('RF Feature Importance ({}) - Top 15:'.format(name))
    print('-' * 55)
    for rank, (_, row) in enumerate(imp_df.head(15).iterrows(), 1):
        marker = ' <-- LF' if row['Feature'] in lf_set else ''
        print('  {:>2}. {:<35} {:.4f}{}'.format(rank, row['Feature'], row['Importance'], marker))

    # Show LF feature ranks
    for lf_feat in sorted(lf_set):
        rank = list(imp_df['Feature']).index(lf_feat) + 1
        imp = imp_df[imp_df['Feature'] == lf_feat]['Importance'].values[0]
        print('  -> {} rank: {}, importance: {:.4f}'.format(lf_feat, rank, imp))
    print()

In [ ]:
# NN first-layer weight magnitude for best LF variant
best_lf = best_lf_reg
nn_model = nn_reg_results[best_lf]['model']
feats = FEATURE_SETS[best_lf]

weights = nn_model.layers[0].get_weights()[0]  # Shape: (n_features, 32)
weight_mag = np.mean(np.abs(weights), axis=1)

weight_df = pd.DataFrame({
    'Feature': feats,
    'Weight_Magnitude': weight_mag
}).sort_values('Weight_Magnitude', ascending=False)

lf_set = lf_feature_names[best_lf]
print('NN First-Layer Weight Magnitude ({}) - Top 15:'.format(best_lf))
print('-' * 55)
for rank, (_, row) in enumerate(weight_df.head(15).iterrows(), 1):
    marker = ' <-- LF' if row['Feature'] in lf_set else ''
    print('  {:>2}. {:<35} {:.4f}{}'.format(rank, row['Feature'], row['Weight_Magnitude'], marker))

for lf_feat in sorted(lf_set):
    rank = list(weight_df['Feature']).index(lf_feat) + 1
    wt = weight_df[weight_df['Feature'] == lf_feat]['Weight_Magnitude'].values[0]
    print('  -> {} rank: {}, weight: {:.4f}'.format(lf_feat, rank, wt))

In [ ]:
# Side-by-side bar charts: RF importance and NN weight magnitude
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

rf_model = trad_results[best_lf]['rf_model']
feats = FEATURE_SETS[best_lf]
lf_set = lf_feature_names[best_lf]

# RF importance
imp_df = pd.DataFrame({
    'Feature': feats,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False).head(12)

ax = axes[0]
y_pos = np.arange(len(imp_df))
colors_rf = ['#f39c12' if f in lf_set else 'steelblue' for f in imp_df['Feature']]
ax.barh(y_pos, imp_df['Importance'], color=colors_rf, alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels(imp_df['Feature'], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('RF Feature Importance ({})'.format(best_lf))
ax.grid(True, alpha=0.3, axis='x')

# NN weight magnitude
top_wt = weight_df.head(12)
ax = axes[1]
y_pos = np.arange(len(top_wt))
colors_nn = ['#f39c12' if f in lf_set else '#9b59b6' for f in top_wt['Feature']]
ax.barh(y_pos, top_wt['Weight_Magnitude'], color=colors_nn, alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_wt['Feature'], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Mean |Weight|')
ax.set_title('NN Weight Magnitude ({})'.format(best_lf))
ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('Feature Importance: RF vs NN (load factor features in orange)', fontsize=11)
plt.tight_layout()
plt.show()

## 10. Route-Level Analysis

Route-level performance is examined to determine whether load factor preferentially benefits specific routes. Since load factor is an industry-aggregate feature (one value per month), its impact is expected to be uniform across routes, primarily capturing demand-side seasonality.

In [ ]:
# Route-level R\u00b2 for NN regression: baseline vs each LF variant
df_test = df_clean[test_mask].copy()

# Generate NN predictions for each variant
for name, feats in FEATURE_SETS.items():
    scaler = nn_reg_results[name]['scaler']
    model = nn_reg_results[name]['model']
    X_test_scaled = scaler.transform(df_test[feats].values)
    col_name = 'nn_pred_{}'.format(name.replace(' ', '_').replace('(', '').replace(')', ''))
    df_test[col_name] = model.predict(X_test_scaled, verbose=0).flatten()

pred_cols = [c for c in df_test.columns if c.startswith('nn_pred_')]

print('Route-Level NN Regression R\u00b2:')
print('=' * 100)
header = '{:<22}'.format('Route')
for name in FEATURE_SETS:
    header += '{:>14}'.format(name[:14])
# Add delta columns for LF variants
for name in lf_names:
    header += '{:>10}'.format('d_' + name[:8])
print(header)
print('-' * 100)

route_nn_results = []
for route in sorted(df_test['route'].unique()):
    rd = df_test[df_test['route'] == route]
    row = {'Route': route}
    r2_vals = {}
    for name, col in zip(FEATURE_SETS.keys(), pred_cols):
        r2 = r2_score(rd['delay_rate'], rd[col])
        r2_vals[name] = r2
        row[name] = r2

    line = '{:<22}'.format(route)
    for name in FEATURE_SETS:
        line += '{:>14.4f}'.format(r2_vals[name])
    for name in lf_names:
        delta = r2_vals[name] - r2_vals['Baseline (Lag12)']
        row['d_' + name] = delta
        line += '{:>+10.4f}'.format(delta)
    print(line)
    route_nn_results.append(row)

route_nn_df = pd.DataFrame(route_nn_results)
print()
for name in lf_names:
    improved = (route_nn_df['d_' + name] > 0).sum()
    print('{}: {}/{} routes improved'.format(name, improved, len(route_nn_df)))

In [ ]:
# Delta bar chart: best LF variant vs baseline (per route)
best_delta_col = 'd_' + best_lf_reg
route_nn_df_sorted = route_nn_df.sort_values(best_delta_col, ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
routes = route_nn_df_sorted['Route'].values
deltas = route_nn_df_sorted[best_delta_col].values
colors = ['#27ae60' if d > 0 else '#e74c3c' for d in deltas]

y_pos = np.arange(len(routes))
ax.barh(y_pos, deltas, color=colors, alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels([r.replace('_', ' \u2192 ') for r in routes], fontsize=8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('\u0394 R\u00b2 (NN: {} - Baseline)'.format(best_lf_reg))
ax.set_title('Route-Level NN R\u00b2 Change with Load Factor ({})'.format(best_lf_reg))
ax.grid(True, alpha=0.3, axis='x')

improved = (deltas > 0).sum()
ax.text(0.98, 0.02, '{}/{} routes improved'.format(improved, len(routes)),
        transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# NN vs traditional: differential load factor impact at route level
# For each route, compare NN delta vs RF delta from adding best LF variant

# Get RF predictions for baseline and best LF variant
rf_baseline = trad_results['Baseline (Lag12)']['rf_model']
rf_best_lf = trad_results[best_lf_reg]['rf_model']

df_test['rf_pred_baseline'] = rf_baseline.predict(df_test[FEATURES_BASELINE].values)
df_test['rf_pred_lf'] = rf_best_lf.predict(df_test[FEATURE_SETS[best_lf_reg]].values)

print('NN vs RF: Differential Load Factor Impact (per route):')
print('{:<22} {:>10} {:>10} {:>12}'.format('Route', 'NN \u0394R\u00b2', 'RF \u0394R\u00b2', 'NN-RF'))
print('-' * 56)

nn_better_count = 0
for route in sorted(df_test['route'].unique()):
    rd = df_test[df_test['route'] == route]
    nn_bl_col = pred_cols[0]  # baseline
    nn_lf_col = pred_cols[list(FEATURE_SETS.keys()).index(best_lf_reg)]

    nn_bl_r2 = r2_score(rd['delay_rate'], rd[nn_bl_col])
    nn_lf_r2 = r2_score(rd['delay_rate'], rd[nn_lf_col])
    nn_delta = nn_lf_r2 - nn_bl_r2

    rf_bl_r2 = r2_score(rd['delay_rate'], rd['rf_pred_baseline'])
    rf_lf_r2 = r2_score(rd['delay_rate'], rd['rf_pred_lf'])
    rf_delta = rf_lf_r2 - rf_bl_r2

    diff = nn_delta - rf_delta
    if nn_delta > rf_delta:
        nn_better_count += 1

    print('{:<22} {:>+10.4f} {:>+10.4f} {:>+12.4f}'.format(route, nn_delta, rf_delta, diff))

print()
print('NN benefits more than RF on {}/{} routes'.format(nn_better_count, len(df_test['route'].unique())))

## 11. Summary and Observations

In [ ]:
print('=' * 80)
print('CONCLUSIONS: NEURAL NETWORK FORECASTING WITH LOAD FACTOR')
print('=' * 80)
print()

threshold = 0.005

# 1. Load factor impact on NN
best_nn_r2_delta = nn_reg_results[best_lf_reg]['r2'] - bl_nr['r2']
best_nn_f1_delta = nn_clf_results[best_lf_clf]['f1'] - bl_nc['f1']
best_nn_rmse_delta = nn_reg_results[best_lf_reg]['rmse'] - bl_nr['rmse']

print('1. LOAD FACTOR IMPACT ON NEURAL NETWORK:')
print('   Best regression variant: {}'.format(best_lf_reg))
print('     R\u00b2:   {:.4f} -> {:.4f} ({:+.4f})'.format(
    bl_nr['r2'], nn_reg_results[best_lf_reg]['r2'], best_nn_r2_delta))
print('     RMSE: {:.4f} -> {:.4f} ({:+.4f})'.format(
    bl_nr['rmse'], nn_reg_results[best_lf_reg]['rmse'], best_nn_rmse_delta))
print('   Best classification variant: {}'.format(best_lf_clf))
print('     F1:   {:.4f} -> {:.4f} ({:+.4f})'.format(
    bl_nc['f1'], nn_clf_results[best_lf_clf]['f1'], best_nn_f1_delta))
print()

# 2. Load factor impact on traditional models
print('2. LOAD FACTOR IMPACT ON TRADITIONAL MODELS:')
print('   Ridge R\u00b2:   {:+.4f} (best: {})'.format(d_ridge, best_ridge))
print('   RF R\u00b2:      {:+.4f} (best: {})'.format(d_rf, best_rf))
print('   XGBoost F1: {:+.4f} (best: {})'.format(d_xgb, best_xgb))
print()

# 3. Route-level summary
improved_routes = {}
for name in lf_names:
    improved_routes[name] = (route_nn_df['d_' + name] > 0).sum()
best_lf_routes = max(lf_names, key=lambda n: improved_routes[n])
print('3. ROUTE-LEVEL SUMMARY:')
for name in lf_names:
    print('   {}: {}/{} routes improved'.format(name, improved_routes[name], len(route_nn_df)))
print()

# 4. Comparison with nowcasting (notebook 17b)
print('4. COMPARISON WITH NOWCASTING (notebook 17b):')
print('   Nowcasting NN + LF Raw: R\u00b2 +0.006 (0.529 -> 0.536), F1 +0.029 (0.743 -> 0.772)')
print('   Forecasting NN + best LF: R\u00b2 {:+.4f}, F1 {:+.4f}'.format(
    best_nn_r2_delta, best_nn_f1_delta))
print()

### Observation

- Regression: NN does improve the R2 score 0.4931 -> 0.5086, when compared with Ridge model
   - However, adding load factor as a feature does not improve the performance
- Classification: NN is marginally worse than XGB, F1 score 0.7450 -> 0.7410
   - In both cases, adding load factor as a feature improves F1 score by about 1%
- Overall, just like for the nowcasting model, using NN does not introduce any significant improvement to the prediction
   - This reinforces the observation that linear models have optimally extracted the predictive value from the available data

## 12. Next Step

The next notebook will explore adding 12-month lagged delay rate to the nowcasting models. This is motivated by the success of adding this feature to the forecasting model ([notebook 15a](/notebooks/15a_forecasting_lag12.ipynb)).